# TP 1: LDA/QDA y optimización matemática de modelos

Se puede consultar la introducción teórica en el mono-notebook, se prefiere mantener este lo más chico posible.

In [51]:
# imports
import numpy as np
import numpy.linalg as LA

from base.qda import QDA, TensorizedQDA
from base.cholesky import QDA_Chol1, QDA_Chol2, QDA_Chol3
from utils.bench import Benchmark
from utils.datasets import (get_iris_dataset, get_letters_dataset,
                            get_penguins_dataset, get_wine_dataset,
                            label_encode)


## Ejemplo

In [52]:
# levantamos el dataset Wine, que tiene 13 features y 178 observaciones en total
X_full, y_full = get_wine_dataset()

X_full.shape, y_full.shape

((178, 13), (178, 1))

In [53]:
# encodeamos a número las clases
y_full_encoded = label_encode(y_full)

y_full[:5], y_full_encoded[:5]

(array([['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0']], dtype='<U7'),
 array([[0],
        [0],
        [0],
        [0],
        [0]]))

In [54]:
# generamos el benchmark
# observar que son valores muy bajos de runs para que corra rápido ahora
b = Benchmark(
    X_full, y_full_encoded,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


In [55]:
# bencheamos un par
to_bench = [QDA]

for model in to_bench:
    b.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [56]:
# como es una clase, podemos seguir bencheando más después
b.bench(TensorizedQDA)

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [57]:
# hacemos un summary
b.summary()

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb
model,,,,,,,,,
QDA,0.361813,0.119873,2.938276,0.188133,0.982407,0.018379,0.000576,0.007605,0.000390
TensorizedQDA,0.414864,0.376631,1.371061,1.008750,0.982593,0.018379,0.000717,0.011986,0.000318


In [58]:
# son muchos datos! nos quedamos con un par nomás
summ = b.summary()

# como es un pandas DataFrame, subseteamos columnas fácil
summ[['train_median_ms', 'test_median_ms','mean_accuracy']]

,train_median_ms,test_median_ms,mean_accuracy
model,,,
QDA,0.361813,2.938276,0.982407
TensorizedQDA,0.414864,1.371061,0.982593


In [59]:
# podemos setear un baseline para que fabrique columnas de comparación
summ = b.summary(baseline='QDA')

summ

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,,,,,,,
QDA,0.361813,0.119873,2.938276,0.188133,0.982407,0.018379,0.000576,0.007605,0.000390,1.000000,1.000000,1.0,1.000000
TensorizedQDA,0.414864,0.376631,1.371061,1.008750,0.982593,0.018379,0.000717,0.011986,0.000318,0.872124,2.143067,1.0,0.634468


In [60]:
# volvemos a subsetear columnas
summ[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.361813,2.938276,0.982407,1.000000,1.000000,1.0,1.000000
TensorizedQDA,0.414864,1.371061,0.982593,0.872124,2.143067,1.0,0.634468


In [61]:
# Levantamos los datasets Iris, Letters, Penguins y Wine
X_full_iris, y_full_iris = get_iris_dataset()
# X_full_letters, y_full_letters = get_letters_dataset()
# X_full_penguins, y_full_penguins = get_penguins_dataset()
X_full_wine, y_full_wine = get_wine_dataset()

X_full_iris.shape, y_full_iris.shape
# X_full_letters.shape, y_full_letters.shape
# X_full_penguins.shape, y_full_penguins.shape
X_full_wine.shape, y_full_wine.shape

((178, 13), (178, 1))

In [62]:
# Encodeamos a número las clases
y_full_iris_encoded = label_encode(y_full_iris)
y_full_wine_encoded = label_encode(y_full_wine)

y_full_iris[:5], y_full_iris_encoded[:5]
y_full_wine[:5], y_full_wine_encoded[:5]

(array([['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0']], dtype='<U7'),
 array([[0],
        [0],
        [0],
        [0],
        [0]]))

In [63]:
benchmark_iris = Benchmark(
    X_full_iris, y_full_iris_encoded,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

benchmark_wine = Benchmark(
    X_full_wine, y_full_wine_encoded,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 105
Test size rows (approx): 45
Test size fraction: 0.3
Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


In [64]:
# Benchmark de los modemos en el dataset Iris
to_bench = [QDA, TensorizedQDA, QDA_Chol1, QDA_Chol2, QDA_Chol3]

for model in to_bench:
    benchmark_iris.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol1 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol1 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol2 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol2 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol3 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol3 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [65]:
for model in to_bench:
    benchmark_wine.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol1 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol1 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol2 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol2 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol3 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol3 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [66]:
summary_iris = benchmark_iris.summary(baseline='QDA')
summary_iris[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.355163,2.375861,0.970667,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,0.343970,0.899428,0.971111,1.032541,2.641524,1.010785,0.918843
QDA_Chol1,0.405128,1.472854,0.972889,0.876669,1.613100,0.949918,0.952842
QDA_Chol2,0.345851,4.279066,0.972222,1.026925,0.555229,0.967218,0.913306
QDA_Chol3,0.344311,1.508339,0.977333,1.031518,1.575151,0.971346,0.987964


In [67]:
summary_wine = benchmark_wine.summary(baseline='QDA')
summary_wine[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.360994,2.960707,0.982407,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,0.390516,1.328354,0.982593,0.924403,2.228854,1.002491,0.641948
QDA_Chol1,0.444002,1.801218,0.985741,0.813047,1.643725,1.012817,0.990668
QDA_Chol2,0.357353,5.171477,0.983333,1.010190,0.572507,1.023522,0.946726
QDA_Chol3,0.368069,1.805204,0.986111,0.980781,1.640096,1.032051,1.009510


# Consigna QDA

## Tensorización

### 1) Diferencias entre `QDA`y `TensorizedQDA`

1. ¿Sobre qué paraleliza `TensorizedQDA`? ¿Sobre las $k$ clases, las $n$ observaciones a predecir, o ambas?

> **R:** Paraleliza únicamente sobre las $k$ clases, ya que la cadena de llamadas es: `predict(X) -> predict_one(X) -> predict_log_conditionals(X)` donde X($p$, 1) es una sola observación. La iteración sobre las $n$ observaciones a predecir se sigue haciendo en `fiuba-ceia-amia-tp/base/bayesian.py` en la línea 34.

2. Analizar los shapes de `tensor_inv_covs` y `tensor_means` y explicar paso a paso cómo es que `TensorizedQDA` llega a predecir lo mismo que `QDA`.
> **R:** La forma de `tensor_inv_covs` es ($k$, $p$, $p$) y la forma de `tensor_means` es ($k$, $p$, 1). Para predecir, `TensorizedQDA` hace un producto punto entre `tensor_inv_covs` y `tensor_means`. La razón principal por la que predicen lo mismo está en observar el código de `_predict_log_conditional` y `_predict_log_conditionals`: ambas devuelven el cálculo de: $$f_j(x) = \frac{1}{(2 \pi)^\frac{p}{2} \cdot |\Sigma_j|^\frac{1}{2}} e^{- \frac{1}{2}(x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j)}$$ La única diferencia es que `QDA` lo hace para una clase a la vez, y `TensorizedQDA` lo hace para todas las clases en paralelo. En esta misma línea, en ambas clases el método `_predict_one` devuelve el índice de la clase que maximiza la probabilidad condicional a posteriori: $$log P(G=k) + log P(x|G=k)$$ Por lo tanto, ambas clases predicen lo mismo, la diferencia es únicamente en performance, siendo `TensorizedQDA` más rápido al paralelizar sobre las clases.

In [68]:
from base.qda import TensorizedQDA
from utils.datasets import get_wine_dataset, label_encode
from sklearn.model_selection import train_test_split

X, y = get_wine_dataset()
y = label_encode(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

# transponer porque la clase espera (n_feat, n_obs)
X_train = X_train.T
y_train = y_train.T

model = TensorizedQDA()
model.fit(X_train, y_train)

print('tensor_inv_cov:', model.tensor_inv_cov.shape)
print('tensor_means:  ', model.tensor_means.shape)


tensor_inv_cov: (3, 13, 13)
tensor_means:   (3, 13, 1)


3. Implementar el modelo `FasterQDA` (se recomienda heredarlo de `TensorizedQDA`) de manera de eliminar el ciclo for en el método predict.
> R: El código implementado se encuentra en [qda.py](base/qda.py#L47-L58).

4. Mostrar dónde aparece la mencionada matriz de $n \times n$, donde $n$ es la cantidad de observaciones a predecir.
> **R:** La matriz de $n \times n$ aparece en [qda.py](base/qda.py#L51), donde se calcula `inner_prod`. En esta línea, `unbiased_X` tiene forma ($k$, $p$, $n$) y `tensor_inv_cov` tiene forma ($k$, $p$, $p$). Al hacer el producto punto entre estas dos matrices, se obtiene una matriz de forma ($k$, $n$, $n$).

5. Demostrar que $$diag(A \cdot B) = \sum_{cols} A \odot B^T = np.sum(A \odot B^T, axis=1)$$ es decir, que se puede "esquivar" la matriz de $n \times n$ usando matrices de $n \times p$. También se puede usar, de forma equivalente $$np.sum(A^T \odot B, axis=0).T$$ queda a preferencia del alumno cuál usar.
> **R:** La demostración parte de como está definido el producto matricial $$C = A \cdot B$$ donde $c_{ij} = \sum_{j} a_{ik} b_{kj}$. Para obtener la diagonal de $C$, se necesita que $i=j$, por lo tanto cada elemento de la diagonal tiene la forma $c_{ii} = \sum_{k} a_{ik} b_{ki}$. Luego, $C' = A \odot B^T$ es un producto elemento a elemento, donde cada elemento tiene la forma $c'_{ik} = a_{ik} b^T_{ik} = a_{ik} b_{ki}$. Vemos entonces la relación, que la sumatoria de los elementos de la fila $i$ de $C'$ es exactamente igual a $c_{ii}$, quedando demostrado que $$diag(A \cdot B) = \sum_{cols} A \odot B^T = np.sum(A \odot B^T, axis=1)$$

In [69]:
import numpy as np

n = 3

matrix_A = np.random.rand(n, n)
matrix_B = np.random.rand(n, n)

diag_product = np.diag(matrix_A @ matrix_B)

print('Diagonal del producto de A y B:', diag_product)

sum_cols = np.sum(matrix_A * matrix_B.T, axis=1)
print('Suma de columnas de A * B^T:', sum_cols)

sum_cols_equiv = np.sum(matrix_A.T * matrix_B, axis=0).T
print('Suma de columnas de A^T * B (equivalente):', sum_cols_equiv)

Diagonal del producto de A y B: [0.37264242 0.78558563 1.14000227]
Suma de columnas de A * B^T: [0.37264242 0.78558563 1.14000227]
Suma de columnas de A^T * B (equivalente): [0.37264242 0.78558563 1.14000227]


6. Utilizar la propiedad antes demostrada para reimplementar la predicción del modelo `FasterQDA` de forma eficiente en un nuevo modelo `EfficientQDA`.
> R: El código implementado se encuentra en [qda.py](base/qda.py#L59-L71).

7. Comparar la performance de las 4 variantes de QDA implementadas hasta ahora (no Cholesky) ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?
> **R:** Se coincide con lo esperado: `EfficientQDA > FasterQDA > TensorizedQDA > QDA` en términos de iteraciones por segundo en cuanto a la inferencia del modelo. Del análisis se desprende que todos los modelos se entrenan más o menos a la misma velocidad, la eficiencia es únicamente en términos de predicción. Esto tiene sentido, ya que las clases implementadas solo aplican sobreescriben los métodos de inferencia.

In [70]:
from base.qda import FasterQDA, EfficientQDA

tensorized_bench = [QDA, TensorizedQDA, FasterQDA, EfficientQDA]

for model in tensorized_bench:
    benchmark_iris.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

FasterQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

FasterQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [71]:
tensorized_summary_iris = benchmark_iris.summary(baseline='QDA')
tensorized_summary_iris[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.325629,2.224455,0.972667,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,0.353669,0.940449,0.972222,0.920717,2.365313,1.013541,0.946595
QDA_Chol1,0.405128,1.472854,0.972889,0.803768,1.510303,0.970636,0.981620
QDA_Chol2,0.345851,4.279066,0.972222,0.941530,0.519846,0.988314,0.940890
QDA_Chol3,0.344311,1.508339,0.977333,0.945741,1.474772,0.992532,1.017803
FasterQDA,0.331712,0.063712,0.971556,0.981662,34.914505,1.023738,0.067425
EfficientQDA,0.333110,0.054090,0.974889,0.977543,41.125077,1.019890,0.195747


In [72]:
for model in tensorized_bench:
    benchmark_wine.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

FasterQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

FasterQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [73]:
tensorized_summary_wine = benchmark_wine.summary(baseline='QDA')
tensorized_summary_wine[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.366034,2.955592,0.982222,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,0.374147,1.260617,0.984444,0.978317,2.344561,1.015914,0.649944
QDA_Chol1,0.444002,1.801218,0.985741,0.824398,1.640885,0.992268,1.003008
QDA_Chol2,0.357353,5.171477,0.983333,1.024294,0.571518,1.002755,0.958519
QDA_Chol3,0.368069,1.805204,0.986111,0.994474,1.637262,1.011111,1.022085
FasterQDA,0.366145,0.079671,0.986667,0.999698,37.097695,0.998312,0.071393
EfficientQDA,0.426885,0.077525,0.985556,0.857455,38.124372,0.990165,0.103893


## Cholesky

Hasta ahora todos los esfuerzos fueron enfocados en realizar una predicción más rápida. Los tiempos de entrenamiento (teóricos al menos) siguen siendo los mismos o hasta (minúsculamente) peores, dado que todas las mejoras siguen llamando al método `_fit_params` original de `QDA`.

La descomposición/factorización de [Cholesky](https://en.wikipedia.org/wiki/Cholesky_decomposition#Statement) permite factorizar una matriz definida positiva $A = LL^T$ donde $L$ es una matriz triangular inferior. En particular, si bien se asume que $p \ll n$, invertir la matriz de covarianzas $\Sigma$ para cada clase impone un cuello de botella que podría alivianarse. Teniendo en cuenta que las matrices de covarianza son simétricas y salvo degeneración, definidas positivas, Cholesky como mínimo debería permitir invertir la matriz más rápido.

*Nota: observar que calcular* $A^{-1}b$ *equivale a resolver el sistema* $Ax=b$.

### 3) Diferencias entre implementaciones de `QDA_Chol`

8. Si una matriz $A$ tiene fact. de Cholesky $A=LL^T$, expresar $A^{-1}$ en términos de $L$. ¿Cómo podría esto ser útil en la forma cuadrática de QDA?

Factorización:

①: $(A B)^{-1} = B^{-1} A^{-1}$

②: $(A B)^T = B^T A^T$

③:

$L L^{-1} = I$

$(L L^{-1})^T = I^T$

&②: $(L^{-1})^T L^T = I^T = I$

$(L^{-1})^T = (L^T)^{-1}$

④:

$A = L L^T$

$A^{-1} = (L L^T)^{-1}$

&①: $A^{-1} = (L^T)^{-1} L^{-1}$

&③: $A^{-1} = (L^{-1})^T * L^{-1}$

Utilidad: en vez de invertir $\Sigma$ por $O(p^3)$, se factoriza con Cholesky por también $O(p^3)$ pero más rápido por ser simétrica, después:
- La evaluación de la forma cuadrática usa $L$ triangular - resolver sistemas es $O(p^2)$ en vez de $O(p^3)$
- El determinante $|\Sigma|$ se obtiene del producto de la diagonal de $L$
- Todo es numéricamente más estable porque $L$ está mejor condicionada que $\Sigma$

9. Explicar las diferencias entre `QDA_Chol1`y `QDA` y cómo `QDA_Chol1` llega, paso a paso, hasta las predicciones.

`QDA`:

- $\Sigma \to \Sigma^{-1}$ via `inv`

`QDA_Chol1`:

- $\Sigma → L^{-1}$ via `cholesky` y `inv`, después evitando operar con la matriz llena $\Sigma^{-1}$

- $|\Sigma^{-1}| \to \prod_{} diag(L^{-1})$

`QDA_Chol1` paso a paso:

- En fit: calcula $\Sigma$ con `cov`, la factoriza con `cholesky` en L, después invierte L con `inv` y guarda $L^{-1}$
- En predict:
  - $unbiased\_x = x - \mu$
  - $y = L^{-1} unbiased\_x$
  - $log|\Sigma^{-1}| = log(\prod_{} diag(L^{-1}))$, porque $|L^{-1}|^2 = (\prod_{} diag(L^{-1}))^2$
  - $||y||^2 = (y^2).sum()$
- Devuelve $log(\prod_{} diag(L^{-1})) - 0.5 * ||y||^2$

10. ¿Cuáles son las diferencias entre `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3`?

`QDA_Chol1`:

- Fit: $\Sigma → L^{-1}$ via `cholesky` y `inv` # lento, no optimizado para la triangulidad
- Predict: $L^{-1} (x - \mu)$ # medio, multiplicar

`QDA_Chol2`:

- Fit: $\Sigma → L$ via `cholesky` # rapido, sin invertir
- Predict: $solve\_triangular(L, x - \mu)$ via `solve_triangular` # lento, resolver

`QDA_Chol3`:

- Fit: $\Sigma → L^{-1}$ via `cholesky` y `dtrtri` # medio, optimizado
- Predict: $L^{-1} (x - \mu)$ # medio, multiplicar


11. Comparar la performance de las 7 variantes de QDA implementadas hasta ahora ¿Qué se observa?¿Hay alguna de las implementaciones de `QDA_Chol` que sea claramente mejor que las demás?¿Alguna que sea peor?

|                   | Time (train): theory / fact       | Time (test): theory / fact                    | Memory (train): theory / fact | Memory (test): theory / fact     |
| ----------------- | --------------------------------- | --------------------------------------------- | ----------------------------- | -------------------------------- |
| **QDA**           | medio / medio                     | medio / medio                                 | media / media                 | media / media                    |
| **TensorizedQDA** | medio / medio                     | rápido (optimizaciones de bajo nivel) / medio | media / media                 | media / baja                     |
| **FasterQDA**     | medio / medio                     | rápido (observaciones juntos) / muy rapido    | alta (matriz n*n) / media     | media / muy baja                 |
| **EfficientQDA**  | medio / medio                     | rápido (sin n*n) / muy rapido                 | media / media                 | media / muy baja                 |
| **QDA_Chol1**     | lento (inv) / lento               | medio / medio                                 | media / media                 | media / media                    |
| **QDA_Chol2**     | muy rápido (no invierte) / rapido | lento (resuelve cada vez) / muy lento         | media / media                 | media / media                    |
| **QDA_Chol3**     | rápido (dtrtri) / medio           | medio / medio                                 | media / media                 | media / media                    |

- Hay una claramente mejor? En general `EfficientQDA` debería ganar en velocidad de test sin sacrificar memoria. `QDA_Chol2` gana en velocidad de fit pero pierde en predict.

- Alguna peor? `FasterQDA` puede explotar en memoria si n es grande (por la matriz n×n). `QDA_Chol1` es lo peor de ambos mundos: inversión cara en fit con `inv` (que no aprovecha triangularidad) y predict igual que `QDA_Chol3`.

- Se condice con lo esperado? Más o menos.

### 4) Optimización

12. Implementar el modelo `TensorizedChol` paralelizando sobre clases/observaciones según corresponda. Se recomienda heredarlo de alguna de las implementaciones de `QDA_Chol`, aunque la elección de cuál de ellas queda a cargo del alumno según lo observado en los benchmarks de puntos anteriores.

> ?

13. Implementar el modelo `EfficientChol` combinando los insights de `EfficientQDA` y `TensorizedChol`. Si se desea, se puede implementar `FasterChol` como ayuda, pero no se contempla para el punto.

> ?

14. Comparar la performance de las 9 variantes de QDA implementadas ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

> ?

## Importante:

Las métricas que se observan al realizar benchmarking son muy dependientes del código que se ejecuta, y por tanto de las versiones de las librerías utilizadas. Una forma de unificar esto es utilizando un gestor de versiones y paquetes como _uv_ o _Poetry_, otra es simplemente usando una misma VM como la que provee Colab.

**Cada equipo debe informar las versiones de Python, NumPy y SciPy con que fueron obtenidos los resultados. En caso de que sean múltiples, agregar todos los casos**. La siguiente celda provee una ayuda para hacerlo desde un notebook, aunque como es una secuencia de comandos también sirve para consola.

In [74]:
%%bash
python --version
uv pip freeze | grep -E "scipy|numpy"

Python 3.12.13
numpy==2.3.1
scipy==1.16.0
